# 07 · Exportar el paquete del póster web

El póster interactivo (`poster/streamlit_app.py`) se publica en **Streamlit Community Cloud**, que no tiene acceso al
workspace de Databricks. Este notebook toma las tablas Gold ya calculadas en la nube y arma un paquete liviano
(pocos MB) en `/Volumes/.../raw/poster/`:

| Archivo | Contenido |
|---|---|
| `h3_poster.parquet` | Un registro por hexágono H3: NDVI por año, pérdida oficial, perturbación, covariables, clusters LISA, Gi* y probabilidad del modelo |
| `capas/*.geojson` | Límite departamental, ríos, vía, ANP, zonas de amortiguamiento y poblados, recortados al área y simplificados |
| `resumen.json` | Cifras globales usadas en el póster |
| `modelo/rf_perdida.joblib` | Random Forest entrenado en el notebook 06 (MLflow) |
| `modelo/zona_demo.parquet` | Zona de 5 × 5 km con las 33 variables por píxel, para puntuar en vivo |

`scripts/exportar_resultados.ps1` descarga esta carpeta a `poster/data/` del repositorio; al hacer *push*,
Streamlit Cloud actualiza el póster automáticamente.

In [ ]:
%pip install -q geopandas shapely h3 scikit-learn mlflow
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
import json, shutil, tempfile
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box

OUT = Path(VOL) / "poster"
(OUT / "capas").mkdir(parents=True, exist_ok=True)

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas().pipe(dbx.clean)
cl = spark.table(dbx.tbl("gold_h3_clusters")).toPandas().pipe(dbx.clean)
pr = (spark.table(dbx.tbl("gold_h3_prediccion")).toPandas().pipe(dbx.clean)
      .rename(columns={"n": "n_px_modelo"}))
df = (feat.merge(cl[["h3", "lisa", "gi_z", "hotspot"]], on="h3", how="left")
          .merge(pr[["h3", "prob_modelo", "frac_predicha", "n_px_modelo"]], on="h3", how="left"))

keep = ["h3", "lat", "lon", "departamento", "provincia", "distrito", "n_px_total",
        "frac_perdida_oficial", "tasa_perdida_oficial", "tasa_perdida_hansen", "frac_bosque_y0", "frac_bosque_2025",
        "riesgo_deforestacion", "d_ndvi", "d_pert", "dist_via_km", "dist_rio_km", "dist_cauce_km", "en_anp", "en_za",
        "lisa", "gi_z", "hotspot", "prob_modelo", "frac_predicha", "n_px_modelo"]
keep += [c for c in df.columns if c.startswith(("ndvi_", "pert_")) and c.split("_")[1].isdigit()]
keep = [c for c in dict.fromkeys(keep) if c in df.columns]
df = df[keep].copy()
for c in df.select_dtypes("float").columns:          # precisión suficiente para un mapa, archivo más liviano
    df[c] = df[c].astype("float32").round(4)
df.to_parquet(OUT / "h3_poster.parquet", index=False)
print(f"h3_poster.parquet: {len(df):,} hexágonos, {df.shape[1]} columnas, "
      f"{(OUT / 'h3_poster.parquet').stat().st_size / 1e6:.2f} MB")

In [ ]:
# Capas de contexto recortadas al área de estudio y simplificadas (≈50 m)
VEC = Path(VOL) / "bronze" / "vector"
caja = box(*C.AOI_BBOX_WGS84).buffer(0.03)
capas = {"departamentos": 0.0005, "rios": 0.0003, "vias": 0.0003, "areas_protegidas_sernanp": 0.0005,
         "zonas_amortiguamiento": 0.0005, "poblados": None}
for nombre, tol in capas.items():
    f = VEC / f"{nombre}.geojson"
    if not f.exists():
        print(f"  (no existe {f.name})")
        continue
    g = gpd.read_file(f).to_crs(4326)
    g = g[g.intersects(caja)].copy()
    if tol is not None:
        g["geometry"] = g.geometry.intersection(caja).simplify(tol, preserve_topology=True)
    g = g[~g.geometry.is_empty]
    cols = [c for c in ["name", "anp_nomb", "anp_cate", "departamento", "highway", "fclass"] if c in g.columns]
    g = g[cols + ["geometry"]]
    tmp = Path(tempfile.gettempdir()) / f"{nombre}.geojson"      # GDAL escribe en disco local y se copia al Volume
    g.to_file(tmp, driver="GeoJSON")
    shutil.copyfile(tmp, OUT / "capas" / f"{nombre}.geojson")
    print(f"  {nombre:28s} {len(g):5d} geometrías  {(OUT / 'capas' / f'{nombre}.geojson').stat().st_size / 1e3:8.1f} kB")

## Modelo entrenado para el póster

Se exporta el **mismo Random Forest registrado en MLflow** por el notebook 06 (si no se encuentra, se reentrena con la
misma muestra y parámetros) y una **zona de demostración** de 5 × 5 km con todas las variables por píxel. En Streamlit
Community Cloud el modelo puntúa esa zona en vivo; el procesamiento masivo sigue ocurriendo en Databricks.

In [ ]:
import joblib, sklearn
from pyspark.sql import functions as F

silver = spark.table(dbx.tbl("silver_s2_pixeles"))
anios = sorted(r.year for r in silver.select("year").distinct().collect())
Y0, Y1 = anios[0], anios[-1]                                                # igual que el notebook 06
base = ["b02", "b03", "b04", "b08", "b11", "b12", "ndvi", "nbr", "ndwi", "mndwi", "bsi"]
FEATURES = [f"{c}_{s}" for c in base for s in ("t0", "t1", "d")]          # mismo orden que el notebook 06
s0 = silver.filter(F.col("year") == Y0).select("row", "col", *[F.col(c).alias(f"{c}_t0") for c in base])
s1 = silver.filter(F.col("year") == Y1).select("row", "col", *[F.col(c).alias(f"{c}_t1") for c in base])
gbp = spark.table(dbx.tbl("silver_geobosques_pixeles"))
previos = list(range(1, Y0 - 2000 + 1))
lab = (gbp.filter(F.col("cobertura_gb").isin(2, 3, 4) & ~F.col("perdida_gb").isin(previos)
                  & (F.col("perdida_gb") != C.GEOBOSQUES_NATURAL_2025))
       .select("row", "col", "lon", "lat", F.col("perdida_gb").isin(C.geobosques_codes(Y0, Y1)).cast("int").alias("label")))
data = s0.join(s1, ["row", "col"]).join(lab, ["row", "col"])
for c in base:
    data = data.withColumn(f"{c}_d", F.col(f"{c}_t1") - F.col(f"{c}_t0"))

# 1) Modelo registrado en MLflow por el notebook 06
modelo, origen, meta_run = None, "", {}
try:
    import mlflow, mlflow.sklearn
    runs = mlflow.search_runs(search_all_experiments=True, max_results=1,
                              filter_string="attributes.run_name = 'rf_perdida_bosque_mdd'",
                              order_by=["attributes.start_time DESC"])
    r = runs.iloc[0]
    modelo = mlflow.sklearn.load_model(f"runs:/{r.run_id}/model")
    meta_run = {"umbral": float(r["params.umbral"]), "prevalencia": float(r["params.prevalencia"]),
                "auc_bloques": float(r["metrics.auc_bloques"]), "auc_aleatorio": float(r["metrics.auc_aleatorio"]),
                "mlflow_run_id": r.run_id}
    origen = f"MLflow (run {r.run_id})"
except Exception as e:
    print("No se pudo cargar desde MLflow:", str(e)[:200])

# 2) Respaldo: reentrenar con la misma muestra estratificada y los mismos hiperparámetros
if modelo is None:
    from sklearn.ensemble import RandomForestClassifier
    counts = {r.label: r["count"] for r in data.groupBy("label").count().collect()}
    pi = counts.get(1, 0) / sum(counts.values())
    fr = {k: min(1.0, 60_000 / v) for k, v in counts.items()}
    pdf = data.sampleBy("label", fractions=fr, seed=42).toPandas().pipe(dbx.clean)
    modelo = RandomForestClassifier(n_estimators=120, max_depth=14, min_samples_leaf=20, class_weight="balanced",
                                    n_jobs=-1, random_state=42).fit(pdf[FEATURES].fillna(0).values, pdf.label.values)
    meta_run = {"umbral": 0.331, "prevalencia": pi}
    origen = "reentrenado en el notebook 07 (mismos parámetros que el 06)"
print("Modelo:", origen)

(OUT / "modelo").mkdir(parents=True, exist_ok=True)
tmp = Path(tempfile.gettempdir()) / "rf_perdida.joblib"
joblib.dump(modelo, tmp, compress=3)
shutil.copyfile(tmp, OUT / "modelo" / "rf_perdida.joblib")
meta = {"features": FEATURES, "y0": Y0, "y1": Y1, "origen": origen, "sklearn": sklearn.__version__,
        "numpy": np.__version__, "importancias": dict(zip(FEATURES, modelo.feature_importances_.round(5).tolist())),
        **meta_run}
(OUT / "modelo" / "modelo_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"rf_perdida.joblib: {(OUT / 'modelo' / 'rf_perdida.joblib').stat().st_size / 1e6:.1f} MB | "
      f"scikit-learn {sklearn.__version__} | numpy {np.__version__} | umbral {meta['umbral']:.3f}")

In [ ]:
# Zona de demostración: ventana de 256 × 256 píxeles (5.1 km) con más pérdida oficial
VENT = 256
mejor = (data.groupBy((F.col("row") / VENT).cast("int").alias("r"), (F.col("col") / VENT).cast("int").alias("c"))
             .agg(F.sum("label").alias("perdida"), F.count("*").alias("n"))
             .orderBy(F.desc("perdida")).first())
r0, c0 = mejor.r * VENT, mejor.c * VENT
zona = (data.filter(F.col("row").between(r0, r0 + VENT - 1) & F.col("col").between(c0, c0 + VENT - 1))
            .toPandas().pipe(dbx.clean))
zona["row"], zona["col"] = (zona["row"] - r0).astype("int16"), (zona["col"] - c0).astype("int16")
for c in FEATURES + ["lon", "lat"]:
    zona[c] = zona[c].astype("float32")
zona["label"] = zona["label"].astype("int8")
zona[["row", "col", "lon", "lat", "label"] + FEATURES].to_parquet(OUT / "modelo" / "zona_demo.parquet", index=False)
print(f"Zona demo: {len(zona):,} píxeles de bosque | pérdida oficial {zona.label.sum() * 0.04:,.0f} ha | "
      f"centro {zona.lat.mean():.3f}, {zona.lon.mean():.3f} | "
      f"{(OUT / 'modelo' / 'zona_demo.parquet').stat().st_size / 1e6:.1f} MB")

In [ ]:
# Resumen global (se muestra en el encabezado del póster)
HA_PX = 0.04
bos = df["frac_bosque_y0"] >= 0.5
res = {
    "hexagonos": int(len(df)),
    "hexagonos_bosque": int(bos.sum()),
    "perdida_oficial_ha": float((df["frac_perdida_oficial"] * df["n_px_total"]).sum() * HA_PX),
    "perdida_modelo_ha": float((df["prob_modelo"] * df["n_px_modelo"]).sum() * HA_PX),
    "frentes_lisa": int((df["lisa"] == "Pérdida-Pérdida (frente activo)").sum()),
    "calientes_99": int((df["hotspot"] == "Caliente 99%").sum()),
    "departamentos": sorted(df["departamento"].dropna().unique().tolist()) if "departamento" in df else [],
}
(OUT / "resumen.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(res, ensure_ascii=False, indent=2))

Siguiente paso, en la computadora local: `.\scripts\exportar_resultados.ps1` (copia `poster/` del Volume a
`poster/data/`), luego `git add`, `commit` y `push`. Streamlit Community Cloud vuelve a desplegar el póster solo.